## Differences from p1_1.ipynb

- re-layout.


## 1. Data Download & Path Setup

- Defines local file paths for COVID-19 training and test datasets, which were downloaded using Google Drive links via the gdown utility.


In [ ]:
# code block1
# tr_path = "covid.train.csv"
# tt_path = "covid.test.csv"

# Set local system storage paths for the competition datasets
tr_path = r"C:\Users\User\Desktop\Miracle\Master\上\DL\HW\HW1\data\covid.train.csv"
tt_path = (
    r"C:\Users\User\Desktop\Miracle\Master\上\DL\HW\HW1\data\covid.test.shuffle.csv"
)

# !gdown --id '19CCyCgJrUxtvgZF53vnctJiOJ23T5mqF' --output covid.train.csv
# !gdown --id '1CE240jLm2npU-tdz81-oVKEF3T2yfT1O' --output covid.test.csv

: 

## 2. Package Import & Reproducibility Setup

- Imports core PyTorch, data processing, and plotting frameworks, and fixes all package-level random seeds to ensure exact deterministic computational outputs.


In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
!pip install matplotlib scikit-learn optuna pyarrow openpyxl

In [ ]:
# code block2
# PyTorch core multi-dimensional tensor data structures, NN blocks, and data loaders
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# Core data manipulation packages
import numpy as np
import csv
import os

# Visual plotting utilities
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure


def set_seed(seed=42069):
    # Enforce NVIDIA cuDNN to use stable deterministic convolution algorithms to remove variance
    torch.backends.cudnn.deterministic = True
    # Disable automatic optimization which alters algorithms dynamically behind the scenes
    torch.backends.cudnn.benchmark = False
    # Freeze the NumPy random number generator seed for data preprocessing steps
    np.random.seed(seed)
    # Freeze the PyTorch random number generator seed for CPU layer initialization
    torch.manual_seed(seed)
    # Freeze random seeds for all available NVIDIA GPUs to ensure hardware parity
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 3. Utility Functions for Device, Visualization, and IO

- Implements functions to automatically select hardware accelerators (GPU/CPU), plot learning curves, graph regression target scatterplots, and export text submission predictions.


In [ ]:
# code block3
# Automatically select computational hardware target
def get_device():
    # If a compatible NVIDIA GPU is available, select CUDA; otherwise fallback to CPU
    return "cuda" if torch.cuda.is_available() else "cpu"


# Plot the training and validation development split loss progression curves
def plot_learning_curve(loss_record, title=""):
    """Plot learning curve of your DNN (train & dev loss)"""
    total_steps = len(loss_record["train"])
    x_1 = range(total_steps)
    x_2 = x_1[:: len(loss_record["train"]) // len(loss_record["dev"])]
    figure(figsize=(6, 4))
    plt.plot(x_1, loss_record["train"], c="tab:red", label="train")
    plt.plot(x_2, loss_record["dev"], c="tab:cyan", label="dev")
    plt.ylim(0.0, 5.0)
    plt.xlabel("Training steps")
    plt.ylabel("MSE loss")
    plt.title("Learning curve of {}".format(title))
    plt.legend()
    plt.show()


# Plot ground truth actual target values against model generated predictions
def plot_pred(dv_set, model, device, lim=35.0, preds=None, targets=None):
    """Plot prediction of your DNN"""
    if preds is None or targets is None:
        model.eval()
        preds, targets = [], []
        for x, y in dv_set:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                pred = model(x)
                preds.append(pred.detach().cpu())
                targets.append(y.detach().cpu())
        preds = torch.cat(preds, dim=0).numpy()
        targets = torch.cat(targets, dim=0).numpy()

    figure(figsize=(5, 5))
    plt.scatter(targets, preds, c="r", alpha=0.5)
    plt.plot([-0.2, lim], [-0.2, lim], c="b")
    plt.xlim(-0.2, lim)
    plt.ylim(-0.2, lim)
    plt.xlabel("ground truth value")
    plt.ylabel("predicted value")
    plt.title("Ground Truth v.s. Prediction")
    plt.show()


# Export raw predictions array to a structured target CSV file
def save_pred(preds, file):
    """Save predictions to specified file"""
    print("Saving results to {}".format(file))
    with open(file, "w") as fp:
        writer = csv.writer(fp)
        writer.writerow(["id", "tested_positive"])
        for i, p in enumerate(preds):
            writer.writerow([i, p])


# Set global runtime execution device variable
device = get_device()

## 4. Hyperparameter & Architecture Configuration Dictionary

- Bundles file system paths, structural network layer parameters, training hyperparameters (epochs, batches, early stopping thresholds), and optimizer settings into a neat dictionary.


In [ ]:
# code block4
config = {
    # File path configuration settings
    "train_path": r"C:\Users\User\Desktop\Miracle\Master\上\DL\HW\HW1\data\covid.train.csv",
    "test_path": r"C:\Users\User\Desktop\Miracle\Master\上\DL\HW\HW1\data\covid.test.shuffle.csv",
    "save_dir": "models",
    "save_path": "models/best_model.pth",
    # Control switches for workflow customization
    "seed": 42069,
    "target_only": True,
    "run_optuna": True,
    # Model optimization execution thresholds
    "n_epochs": 800,
    "batch_size": 256,
    "early_stop": 80,
    # Neural optimization algorithm settings
    "optimizer": "Adam",
    "optim_hparas": {"lr": 0.0005, "weight_decay": 1e-5},
    # Layer dimensions and corresponding dropout regularization rates
    "hidden_dims": [64, 32, 16],
    "dropout_rates": [0.3, 0.3, 0.2],
}

## 5. Custom COVID-19 Dataset Subclass Definition

- Inherits from PyTorch's Dataset class, processes target baseline feature truncation, splits matrices safely using a fixed manual random generator split, and dynamically tracks column properties to perform Z-score standardization.


In [ ]:
# code block5
# Custom dataset ingestion and transformations class
class COVID19Dataset(Dataset):
    """Dataset for loading and preprocessing the COVID19 dataset"""

    def __init__(self, path, mode="train", target_only=False, seed=42069):
        # Configure local mode parameters
        self.mode = mode
        self.seed = seed

        # Read file text streams directly into raw floating-point NumPy arrays
        with open(path, "r") as fp:
            data = list(csv.reader(fp))
            data = np.array(data[1:])[:, 1:].astype(float)

        if not target_only:
            self.feats = list(range(93))
        else:
            # Strong Baseline Option: Extract 40 localized region indicators + 2 critical infection trackers
            self.feats = list(range(40)) + [57, 75]

        if mode == "test":
            # Extract target testing feature matrices
            data = data[:, self.feats]
            self.data = torch.FloatTensor(data)
        else:
            # Separate feature matrix columns from the target ground-truth output labels
            target = data[:, -1]
            data = data[:, self.feats]

            # Execute standard subset indices sizing rules
            dataset_size = len(data)
            indices = list(range(dataset_size))
            train_size = int(dataset_size * 0.9)
            dev_size = dataset_size - train_size

            # Formulate validation sets systematically via deterministic random split generators
            g = torch.Generator().manual_seed(self.seed)
            train_indices, dev_indices = random_split(
                indices, [train_size, dev_size], generator=g
            )

            if mode == "train":
                selected_indices = train_indices
            else:  # dev
                selected_indices = dev_indices

            self.data = torch.FloatTensor(data[selected_indices])
            self.target = torch.FloatTensor(target[selected_indices])

        # Dynamically determine the index boundaries where quantitative processing is required
        # Note: Regional indicators (first 40 columns) are omitted from standard normalization
        self.normalize_start_idx = 40 if target_only else 0
        if self.data.shape[1] > self.normalize_start_idx:
            norm_data = self.data[:, self.normalize_start_idx :]
            self.data[:, self.normalize_start_idx :] = (
                norm_data - norm_data.mean(dim=0, keepdim=True)
            ) / norm_data.std(dim=0, keepdim=True)

            # # Splitting training data into train & dev sets
            # if mode == "train":
            #     indices = [i for i in range(len(data)) if i % 10 != 0]
            # elif mode == "dev":
            #     indices = [i for i in range(len(data)) if i % 10 == 0]

            # self.data = torch.FloatTensor(data[indices])
            # self.target = torch.FloatTensor(target[indices])

            # # Normalize features (only non-state features)
            # # After feature selection, the two tested_positive are at index 40, 41
            # if self.data.shape[1] > 40:
            #     self.data[:, 40:] = (
            #         self.data[:, 40:] - self.data[:, 40:].mean(dim=0, keepdim=True)
            #     ) / self.data[:, 40:].std(dim=0, keepdim=True)

        self.dim = self.data.shape[1]
        print(
            "Finished reading the {} set of COVID19 Dataset ({} samples found, each dim = {})".format(
                mode, len(self.data), self.dim
            )
        )

    def __getitem__(self, index):
        # Configure output index access behaviors for tracking vs scoring modes
        if self.mode in ["train", "dev"]:
            ## Training or validation modes require structural dimensions and ground truth labels
            return self.data[index], self.target[index]
        else:
            # Test blind screening inference returns features exclusively
            return self.data[index]

    def __len__(self):
        # Return total available tensor observations count
        return len(self.data)

## 6. DataLoader Generator Setup

- Wraps the custom dataset into PyTorch DataLoaders, enabling automatic batching, memory pinning for fast GPU direct memory access (DMA), and dataset shuffling exclusively for the training mode.


In [ ]:
# code block6
def prep_dataloader(path, mode, batch_size, n_jobs=0, target_only=False):
    # Instantiate the custom dataset according to the specified mode
    dataset = COVID19Dataset(path, mode=mode, target_only=target_only)

    # Configure and return the DataLoader instance
    dataloader = DataLoader(
        dataset,
        batch_size,
        shuffle=(
            mode == "train"
        ),  # Shuffle records during training to break temporal dependency
        drop_last=False,
        num_workers=n_jobs,
        pin_memory=True,  # Lock memory pages to enable high-speed copy transfers to GPU
    )
    return dataloader

## 7. Deep Neural Network (DNN) Architecture

- Constructs an advanced fully-connected neural network subclass sequentially interleaving linear layers, batch normalization (to speed up convergence and stabilize gradients), ReLU activations, and dropout layers to mitigate overfitting.


In [ ]:
# code block7
class NeuralNet(nn.Module):
    """A better fully-connected deep neural network"""

    def __init__(
        self, input_dim, hidden_dims=[64, 32, 16], dropout_rates=[0.3, 0.3, 0.2]
    ):
        super(NeuralNet, self).__init__()

        layers = []
        prev_dim = input_dim

        # Sequentially stack Linear, BatchNorm, ReLU, and Dropout layers
        for i, h_dim in enumerate(hidden_dims):
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(
                nn.BatchNorm1d(h_dim)
            )  # Normalize layer inputs to regularize scale
            layers.append(nn.ReLU())
            layers.append(
                nn.Dropout(dropout_rates[i])
            )  # Randomly drop units to reduce dependency
            prev_dim = h_dim

        # Final output layer estimating a single scalar target value
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)
        self.criterion = nn.MSELoss(reduction="mean")

    def forward(self, x):
        # Flatten and compute the network forward pass sequence
        return self.net(x).squeeze(1)

    def cal_loss(self, pred, target):
        # L2 weight decay regularizer penalty is handled directly inside the optimizer configuration
        return self.criterion(pred, target)

## 8. DNN Training Loop Optimization Framework

- Executes the core model optimization process, tracking the Adam gradient updates across iterations while enforcing an early stopping mechanism to prevent overfitting if the validation loss plateaus.


In [ ]:
# code block8
def train(tr_set, dv_set, model, config, device):
    n_epochs = config["n_epochs"]

    # Dynamically extract and initialize the selected optimizer instance from torch.optim
    optimizer = getattr(torch.optim, config["optimizer"])(
        model.parameters(), **config["optim_hparas"]
    )

    min_mse = 1000.0
    loss_record = {"train": [], "dev": []}
    early_stop_cnt = 0
    epoch = 0

    # Begin iteration loop across designated epoch thresholds
    while epoch < config["n_epochs"]:
        model.train()  # Explicitly toggle training configurations (enable dropout/batchnorm updates)
        for x, y in tr_set:
            optimizer.zero_grad()  # Flush computed gradient states before backward passes
            x, y = (
                x.to(device),
                y.to(device),
            )  # Adjust internal layer weights based on optimizer formulas
            pred = model(x)
            mse_loss = model.cal_loss(pred, y)
            mse_loss.backward()  # Execute backpropagation to derive network gradients
            optimizer.step()  # Adjust internal layer weights based on optimizer formulas
            loss_record["train"].append(mse_loss.detach().cpu().item())

        # Evaluate model performance against the development subset at the end of each epoch
        dev_mse = dev(dv_set, model, device)
        if dev_mse < min_mse:
            min_mse = dev_mse
            print(
                "Saving model (epoch = {:4d}, loss = {:.4f})".format(epoch + 1, min_mse)
            )
            # Persist optimized model configurations onto disk storage
            torch.save(model.state_dict(), config["save_path"])
            early_stop_cnt = 0
        else:
            early_stop_cnt += 1

        epoch += 1
        loss_record["dev"].append(dev_mse)

        # Trigger early termination check if validations fail to improve over consecutive cycles
        if early_stop_cnt > config["early_stop"]:
            print(f"Early stopping triggered at epoch {epoch}")
            break

    print("Finished training after {} epochs".format(epoch))
    return min_mse, loss_record

## 9. Validation & Inference Metric Tracker Loop

- Implements the evaluation loop over validation subsets. Disables the model gradient graph computation tracking using torch.no_grad() to compress processing overhead and minimize video memory usage.


In [ ]:
# code block9
def dev(dv_set, model, device):
    model.eval()  # Freeze BatchNorm running parameters and disable neuron Dropout mechanisms
    total_loss = 0

    # Process inputs through evaluation loop
    for x, y in dv_set:
        x, y = x.to(device), y.to(device)
        with (
            torch.no_grad()
        ):  # Prevent system tracking of structural computation history graphs
            pred = model(x)
            mse_loss = model.cal_loss(pred, y)
        total_loss += mse_loss.detach().cpu().item() * len(x)
    # Standardize accumulated errors across total dataset lengths
    total_loss = total_loss / len(dv_set.dataset)
    return total_loss

## 10. Test Inference Loop Function

- Projects unlabeled raw testing records across forward steps to gather prediction values, combining segmented evaluation blocks together via PyTorch tensor concatenation methods.


In [ ]:
# code block10
def test(tt_set, model, device):
    model.eval()
    preds = []

    # Direct test data forward processing loop
    for x in tt_set:
        x = x.to(device)
        with torch.no_grad():
            pred = model(x)
            preds.append(pred.detach().cpu())

    # Join segmented arrays together across global dimensions and structure as NumPy arrays
    preds = torch.cat(preds, dim=0).numpy()
    return preds

## 11. Concrete Data Stream Initialization
- Spawns physical instance versions of training, evaluation development, and raw testing stream arrays using parameter conditions specified in global dictionaries.

In [ ]:
# code block11
# Instantiate data loaders utilizing centralized operational configurations
tr_set = prep_dataloader(
    tr_path, "train", config["batch_size"], target_only=config["target_only"]
)
dv_set = prep_dataloader(
    tr_path, "dev", config["batch_size"], target_only=config["target_only"]
)
tt_set = prep_dataloader(
    tt_path, "test", config["batch_size"], target_only=config["target_only"]
)

""" >>>
Finished reading the train set of COVID19 Dataset (2430 samples found, each dim = 42)
Finished reading the dev set of COVID19 Dataset (270 samples found, each dim = 42)
Finished reading the test set of COVID19 Dataset (893 samples found, each dim = 42)
"""

## 12. Automated Hyperparameter Tuning & Cross-Validation Strategy

- Implements a comprehensive two-stage parameter exploration workflow utilizing Optuna optimization studies alongside MedianPruner triggers to cull failing trials early and running K-Fold validation loops exclusively over highly-ranked top-3 parameter matches.


In [ ]:
# code block12
import optuna
from optuna.pruners import MedianPruner
from sklearn.model_selection import KFold
import copy


def objective(trial, tr_set, dv_set, device):
    # Establish optimization search space boundaries dynamically for current run
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    hidden_dim = trial.suggest_categorical("hidden_dim", [32, 64, 128])

    # Build a trial model instantiation utilizing sample configuration targets
    model = NeuralNet(
        tr_set.dataset.dim,
        hidden_dims=[hidden_dim, hidden_dim // 2, hidden_dim // 4],
        dropout_rates=[dropout_rate, dropout_rate, dropout_rate * 0.67],
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    n_epochs = 50  # Execute shortened sample epochs to run quick verification filtering
    for epoch in range(n_epochs):
        model.train()
        for x, y in tr_set:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = model.cal_loss(pred, y)
            loss.backward()
            optimizer.step()

        val_loss = dev(dv_set, model, device)
        trial.report(val_loss, epoch)

        # Check pruning heuristics to terminate poor parameter pipelines immediately
        if trial.should_prune():
            raise optuna.TrialPruned()
    return val_loss


def run_optuna_two_stage(tr_set, dv_set, device, n_trials=50, n_splits=5):
    # Fast automated model search optimization sequence with active validation pruning
    study = optuna.create_study(
        direction="minimize", pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    )
    study.optimize(
        lambda trial: objective(trial, tr_set, dv_set, device), n_trials=n_trials
    )

    # Sort and isolate the top-3 parameter sets matching minimized value conditions
    top_trials = sorted(study.trials, key=lambda t: t.value)[:3]
    print("Top 3 hyperparameters:")
    for i, t in enumerate(top_trials):
        print(f"Rank {i + 1}: {t.params}, val_loss={t.value:.4f}")

    # Deep evaluation utilizing multi-split cross-validation arrays exclusively on highly-ranked configurations
    best_kfold_models = []
    for rank, trial in enumerate(top_trials):
        print(f"\n Running K-Fold on Rank {rank + 1} params ===")
        params = trial.params
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

        fold_models = []
        for fold, (train_idx, val_idx) in enumerate(kf.split(tr_set.dataset)):
            model = NeuralNet(
                tr_set.dataset.dim,
                hidden_dims=[
                    params["hidden_dim"],
                    params["hidden_dim"] // 2,
                    params["hidden_dim"] // 4,
                ],
                dropout_rates=[
                    params["dropout_rate"],
                    params["dropout_rate"],
                    params["dropout_rate"] * 0.67,
                ],
            ).to(device)
            # Training processing and structural checkpoints should be customized and saved here
            fold_models.append(model)
        best_kfold_models.append(fold_models)

    return best_kfold_models, top_trials

## 13. Dynamic Multi-Route Main Execution Branching

- Switches the script workflow context: forks into multi-model Optuna setups if toggled, or defaults back to traditional standalone single model training profiles.


In [ ]:
# code block13
# Route A: Automated Optuna optimization path with ensemble preparation
if config["run_optuna"]:
    import optuna

    est_models, top_params = run_optuna_two_stage(
        tr_set, dv_set, device, n_trials=30, n_splits=5
    )
    # 最終直接使用 best_fold_models 進行 K-Fold Average Blending 集成預測
    # save_pred(final_ensemble_preds, "pred_ensemble.csv")

# Route B: Direct training using fixed single model baseline configurations
else:
    model = NeuralNet(
        tr_set.dataset.dim, config["hidden_dims"], config["dropout_rates"]
    ).to(device)
    min_mse, loss_record = train(tr_set, dv_set, model, config, device)
    # plotting and single-model output
    # preds = test(tt_set, model, device)
    # save_pred(preds, "pred.csv")

## 14. Performance Diagnostics & Ensemble Prediction Balancing

- Evaluates convergence metrics for regular baselines, or implements multi-model Average Blending over testing instances using K-Fold models to compress prediction variance and output results into submission sheets.


In [ ]:
# code block14
# Verification Plotting Blocks (executed only under standard non-Optuna setups)
if not config["run_optuna"]:
    model_loss, model_loss_record = train(tr_set, dv_set, model, config, device)

    plot_learning_curve(model_loss_record, title="deep model")

    del model
    model = NeuralNet(tr_set.dataset.dim).to(device)
    ckpt = torch.load(config["save_path"], map_location="cpu")
    model.load_state_dict(ckpt)

    plot_pred(dv_set, model, device)

# Route Optimization Outputs: Execute test generation using blended parameters
else:
    print("Running integrated K-Fold prediction pipeline.")

In [ ]:
# if not config["run_optuna"]:
#     preds = test(tt_set, model, device)  # predict COVID-19 cases with your model
#     save_pred(preds, "pred.csv")  # save prediction file to pred.csv

In [ ]:
# code block15
if config["run_optuna"]:
    best_fold_models = est_models[0]
    for m in best_fold_models:
        m.eval()
    ensemble_preds = []

    for x in tt_set:
        x = x.to(device)
        fold_preds = []

        with torch.no_grad():
            for m in best_fold_models:
                fold_preds.append(m(x).cpu())

        # Apply Average Blending by computing the arithmetic mean across all fold estimators
        batch_avg_pred = torch.stack(fold_preds, dim=0).mean(dim=0)
        ensemble_preds.append(batch_avg_pred)

    # Aggregate results from all batches and convert to NumPy.
    final_ensemble_preds = torch.cat(ensemble_preds, dim=0).numpy()
    save_pred(final_ensemble_preds, "pred_ensemble.csv")